In [ ]:
import sys
!{sys.executable} -m pip install librosa --break-system-packages

In [ ]:
import pandas as pd
import numpy as np
import librosa
import librosa.display
import os

In [ ]:
df = pd.read_csv('../data/cleaned/fma_emotion_labels.csv')


In [ ]:
 ## Parameters 
SAMPLE_RATE = 22050
DURATION = 30       
N_MELS = 128         
HOP_LENGTH = 512
N_FFT = 2048

In [ ]:
def mp3_to_spectrogram(mp3_path, sr=SAMPLE_RATE, duration=DURATION, dims=(128, 1291)):
    """
    Loads an MP3 file and converts it to a mel spectrogram.
    Returns a 2D numpy array (n_mels x time_frames), or None if the file fails.
    """
    try:
        # Loading the audio
        y, sr = librosa.load(mp3_path, sr=sr, duration=duration, mono=True)

        # Computing mel spectrogram
        mel_spec = librosa.feature.melspectrogram(
            y=y,
            sr=sr,
            n_mels=N_MELS,
            n_fft=N_FFT,
            hop_length=HOP_LENGTH
        )

        # Converting to log scale 
        mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

        # Normalizing to [0, 1]
        mel_spec_norm = (mel_spec_db - mel_spec_db.min()) / (mel_spec_db.max() - mel_spec_db.min())

        return mel_spec_norm.reshape(dims[0], dims[1])

    except Exception as e:
        print(f"Failed to process {mp3_path}: {e}")
        return None

In [ ]:
## Generating and saving spectrograms
spectrograms = []
labels = []
track_ids = []

for _, row in df.iterrows():
    spec = mp3_to_spectrogram(row['mp3_path'])
    if spec is not None:
        spectrograms.append(spec)
        labels.append(row['genre_top'])
        track_ids.append(row['track_id'])

In [ ]:
TARGET_FRAMES = 1292  

def pad_or_truncate(spec, target_frames=TARGET_FRAMES):
    if spec.shape[1] >= target_frames:
        return spec[:, :target_frames]
    else:
        pad_width = target_frames - spec.shape[1]
        return np.pad(spec, ((0, 0), (0, pad_width)), mode='constant')

spectrograms_fixed = np.array([pad_or_truncate(s) for s in spectrograms])
labels = np.array(labels)
track_ids = np.array(track_ids)

print(f"Spectrogram array shape: {spectrograms_fixed.shape}")
print(f"Labels: {np.unique(labels)}")

np.save('../data/cleaned/fma_spectrograms.npy', spectrograms_fixed)
np.save('../data/cleaned/fma_labels.npy', labels)
np.save('../data/cleaned/fma_track_ids.npy', track_ids)

len(spectrograms)